In [ ]:
%load_ext autoreload
%autoreload 2

import os
import numpy as np
import matplotlib.pyplot as plt

from plot_serializer.matplotlib.serializer import MatplotlibSerializer

os.chdir("../../../")

from src.postprocessing.utils import find_absolute_min_max
from src.postprocessing.utils_plots import create_standard_list, sort_list_of_dicts, merge_standard_list, add_legend_right, save_data_raw

plt.style.use("FST.mplstyle")

from matplotlib.font_manager import FontProperties
arial_font = FontProperties(family="Arial", style="italic")

outdirectory = "plots/OFF/"

In [ ]:
general_information = {"author": {"name": "Julius H.P. Breuer", "ORCID": "0000-0002-6226-7208"}, "type": "PhD thesis, Technische Universität Darmstadt, Chair of Fluid Systems", "status": "in preparation", "title": "Algorithmische Systemplanung raumlufttechnischer Anlagen", "figure_created_with": "analyse_OFF.ipynb"}

def save_data(fig, serializer, outname, filedata, caption):
    save_data_raw(fig, serializer, outdirectory, outname, filedata, caption, general_information=general_information, id_result_subfolder = "id_results/")

In [ ]:
pt = 1./72.27
diss_tex_width = 390.0*pt

In [ ]:
# CONTROL STRATEGY NAMES:
CAV = "KVS"
VAVCPC = "KONST/ZENTRAL"
VAVVPC = "VAR/ZENTRAL"
DFCPC = "KONST/VERTEILT"
ONLYDF = "NUR/VERTEILT"
ODSCC = "OPT"

control_strategies_names_ger = [CAV, VAVCPC, VAVVPC, DFCPC, ONLYDF, ODSCC]

control_strategies_names_ger_twocolumn = ["KVS", "KONST/\nZENTRAL", "VAR/\nZENTRAL", "KONST/\nVERTEILT", "OHNE/\nVERTEILT", "OPT"]

control_strategies_names_opt = ["CAV", "VAV-CPC", "VAV-VPC", "DF-CPC", "ONLY-DF", "ODS-CC"]

markerstyle = ["o","D", "s","v","^","P"]

In [ ]:
out_directory = "plots/diss/"

save_plots = input("Save all plots? y/n")
save_plots = True if save_plots.lower() == "y" else False

## LCC Bar charts

In [ ]:
standard_case_bar_chart = True

if standard_case_bar_chart:
    folder = "results/off/combined/max_velocity_5_10ms_max_height_04m/standard_case/"

    ticknames = control_strategies_names_ger_twocolumn

    foldernames = ["CAV", "VAV-CPC", "VAV-VPC", "DF-CPC", "ONLY-DF", "CAV"]
    filename_list = [folder + x for x in foldernames]

    standard_dict_list = create_standard_list(filename_list, "Topology", use_connected_opt=False)

    standard_dict_list

    # Assume colors are in hex RGBA
    colors = ['#000000ff', '#535353ff', '#898989ff', '#b5b5b5ff', '#dcdcdcff', "green"]

    # Compute indices of maximum LCC
    from src.postprocessing.utils import find_strategy_arg_min_max
    arg_max_lcc, _ = find_strategy_arg_min_max(standard_dict_list, "exact_lcc")

    serializer = MatplotlibSerializer()

    fig,ax = serializer.subplots(figsize=(diss_tex_width,diss_tex_width*5/9))

    entry_names = ["duct_costs", "equipment_investment_cost", "sil_costs", "vfc_costs", "fan_costs", "exact_fan_energy_costs"]
    cost_label = ["Kanal", "Regel-Eq.", "Schalldämpfer", "VSR", "Ventilator     ", "Energiekosten"]
    print("added whitespace to Ventilator in order to not have the legend titles cut off")
    for curr_dict in standard_dict_list:
        curr_dict["exact_fan_energy_costs"] = (
            curr_dict["fan_energy_costs"]
            * curr_dict["exact_power_consumption"]
            / curr_dict["power_consumption"]
        )

    entry_names_occurring = []
    cost_label_occurring = []
    for i in range(len(entry_names)):
        if not np.all(curr_dict[entry_names[i]] == 0):
            entry_names_occurring.append(entry_names[i])
            cost_label_occurring.append(cost_label[i])
    entry_names = entry_names_occurring
    cost_label = cost_label_occurring


    # Iterate over strategies
    for idx, curr_dict in enumerate(standard_dict_list):

        # Extract relevant costs at arg_max_lcc index
        costs = [curr_dict[name][arg_max_lcc[idx]]/1e3 for name in entry_names]
        if not costs[1]:
            costs[1] = 0

        # Plot stacked bar
        b_cost = 0
        for i, cost in enumerate(costs):
            if i == len(costs) - 1:
                ax.bar(ticknames[idx], cost, bottom=b_cost, color="white", label=cost_label[i], hatch="xx", edgecolor="black", linewidth=0.3)
            else:
                ax.bar(ticknames[idx], cost, bottom=b_cost, color=colors[i], label=cost_label[i], edgecolor="black", linewidth=0.3)
            b_cost += cost

    # Optional: Customize axes
    ax.set_ylabel("LEBENSZYKLUSKOSTEN in $10^3~$€")

    ax.set_xticks(ticknames, ticknames, fontsize=7.5)
    
    ax = add_legend_right(ax, entry_names)

    # ax.set_ylim(top=9e1)

    # ax.set_xticks(foldernames, foldernames, rotation=90)

    fig.tight_layout()
    if save_plots:
        filedata = [x["filename"][0] for x in standard_dict_list]
        outname = "bar_chart_topo_OFF"
        caption = r"Büroturm -- Topologieoptimierung\\Vergleich der lebenszykluskostenoptimalen Lösungen für verschiedene Regelstrategien. \gls{cav} weist die niedrigsten Lebenszykluskosten auf. Maßgeblich dafür sind die geringeren kapitalbezogenen Kosten der Volumenstromregler, die gegenüber \gls{vavvpc} um \SI{40}{\percent} niedriger sind. Der Energieverbrauch ist jedoch um den Faktor 3 höher. Die Kanalkosten variieren, wie beim Multifunktionsgebäude, nur geringfügig."
        save_data(fig, serializer, outname, filedata, caption)

# NOISE

In [ ]:
standard_case_bar_chart = True


if standard_case_bar_chart:
    # folder = "results/GPZ/combined/max_velocity_5_10ms_max_height_04m/standard_case/"
    folder = "results/off/combined/max_velocity_5_10ms_max_height_04m/standard_case/"
    # 
    foldernames = ["CAV", "VAV-CPC", "VAV-VPC", "DF-CPC", "ONLY-DF", "CAV"]
    filename_list = [folder + x for x in foldernames]

    ticknames = control_strategies_names_ger_twocolumn

    standard_dict_list = create_standard_list(filename_list, "Configuration", use_connected_opt=True)

    # Assume colors are in hex RGBA
    colors = ['#000000ff', '#535353ff', '#898989ff', '#b5b5b5ff', '#dcdcdcff', "green"]

    # Compute indices of maximum LCC
    from src.postprocessing.utils import find_strategy_arg_min_max
    arg_max_lcc, _ = find_strategy_arg_min_max(standard_dict_list, "exact_lcc")

    serializer = MatplotlibSerializer()
    fig,ax = serializer.subplots(figsize=(diss_tex_width,diss_tex_width*5/9))

    entry_names = ["duct_costs", "equipment_investment_cost", "sil_costs", "vfc_costs", "fan_costs", "exact_fan_energy_costs"]
    cost_label = ["Kanal", "Regel-Eq.", "Schalldämpfer", "VSR", "Ventilator     ", "Energiekosten"]
    print("added whitespace to Ventilator in order to not have the legend titles cut off")
    for curr_dict in standard_dict_list:
        curr_dict["exact_fan_energy_costs"] = (
            curr_dict["fan_energy_costs"]
            * curr_dict["exact_power_consumption"]
            / curr_dict["power_consumption"]
        )

    entry_names_occurring = []
    cost_label_occurring = []
    for i in range(len(entry_names)):
        if not np.all(standard_dict_list[0][entry_names[i]] == 0):
            entry_names_occurring.append(entry_names[i])
            cost_label_occurring.append(cost_label[i])
    entry_names = entry_names_occurring
    cost_label = cost_label_occurring


    # Iterate over strategies
    for idx, curr_dict in enumerate(standard_dict_list):

        # Extract relevant costs at arg_max_lcc index
        costs = [curr_dict[name][arg_max_lcc[idx]]/1e3 for name in entry_names]
        if not costs[1]:
            costs[1] = 0

        # Plot stacked bar
        b_cost = 0
        for i, cost in enumerate(costs):
            if i == len(costs) - 1:
                ax.bar(ticknames[idx], cost, bottom=b_cost, color="white", label=cost_label[i], hatch="xx", edgecolor="black", linewidth=0.3)
            else:
                ax.bar(ticknames[idx], cost, bottom=b_cost, color=colors[i], label=cost_label[i], edgecolor="black", linewidth=0.3)
            b_cost += cost

    # Optional: Customize axes
    ax.set_ylabel("LEBENSZYKLUSKOSTEN in $10^3~$€")
    ax.set_xticklabels(ticknames, fontsize=7.5)

    ax = add_legend_right(ax, entry_names)


    # ax.set_ylim(top=9e1)

    fig.tight_layout()
    if save_plots:
        filedata = [x["filename"][0] if x["filename"] else "no file" for x in standard_dict_list]
        outname = "bar_chart_config_off"
        caption = r"Büroturm -- Konfigurationsoptimierung\\Vergleich der Lebenszykluskosten-optimalen Lösungen für verschiedene Regelstrategien. Durch die Integration der Akustik in den Planungsprozess steigen die Lebenszykluskosten nur geringfügig. Für Regelstrategie \gls{onlydf} gibt es jedoch keine Lösung mehr."
        save_data(fig, serializer, outname, filedata, caption)

In [ ]:
standard_case_bar_chart = True
if standard_case_bar_chart:

    folderpath = "results/off/combined/max_velocity_5_10ms_max_height_04m/rooms_identical_noise_limits_pareto/"

    subfolders = ["CAV", "VAV-VPC", "DF-CPC"]

    folders = [folderpath + x + "/" for x in subfolders]

    standard_dict_list = create_standard_list(folders, "Configuration", use_connected_opt=False)
    # curr_dict = merge_standard_list(standard_dict_list)

    standard_dict_list = sort_list_of_dicts(standard_dict_list, "max_noise_value")

    cs_names = [control_strategies_names_ger[0], control_strategies_names_ger[2], control_strategies_names_ger[3], control_strategies_names_ger[-1]]

    markers = [x for idx, x in enumerate(markerstyle) if idx in [0,2,3,-1]]


    colors = ['#000000ff', '#535353ff', '#898989ff', '#b5b5b5ff', '#dcdcdcff', "green"]

    # Plot setup
    serializer = MatplotlibSerializer()
    fig,ax = serializer.subplots(figsize=(0.8*diss_tex_width,diss_tex_width*5/9))

    entry_names = ["duct_costs", "equipment_investment_cost", "sil_costs", "vfc_costs", "fan_costs", "exact_fan_energy_costs"]
    cost_label = ["DUCT INVEST", "EQ INVEST", "SILENCER INVEST", "VFC INVEST", "FAN INVEST", "FAN ENERGY"]

    for idx, curr_dict in enumerate(standard_dict_list):
        # Compute exact fan energy costs
        curr_dict["exact_fan_energy_costs"] = (
            curr_dict["fan_energy_costs"]/1e3
            * curr_dict["exact_power_consumption"]
            / curr_dict["power_consumption"]
        )

        ax.plot(curr_dict["max_noise_value"],curr_dict["exact_lcc"]/1e3, "-", color="k", markeredgecolor="k", zorder=-0.5)
        ax.plot(curr_dict["max_noise_value"],curr_dict["exact_lcc"]/1e3, color="k", marker=markers[idx],markersize=5,markerfacecolor="white", markeredgecolor="k", label=cs_names[idx])
    
        ax.plot([curr_dict["max_noise_value"][0]]*2, [curr_dict["exact_lcc"][0]/1e3, 60e1], color="black", zorder=-1)
        ax.plot([curr_dict["max_noise_value"][-1],60], [curr_dict["exact_lcc"][-1]/1e3]*2, color="black", zorder=-1)

        if idx == 0:
            ax.fill_between(curr_dict["max_noise_value"], 30e1, curr_dict["exact_lcc"]/1e3,color="lightgrey",zorder=-2)
            ax.fill_between([39,curr_dict["max_noise_value"][0]], 30e1, 60e1,color="lightgrey",zorder=-2)
            ax.fill_between([curr_dict["max_noise_value"][-1],56], 30e1, curr_dict["exact_lcc"][-1]/1e3,color="lightgrey",zorder=-2)


            ax.plot([curr_dict["max_noise_value"][-1], 56],[curr_dict["exact_lcc"][-1]/1e3]*2,  color="black", zorder=-1)



    ax.plot([39.5,40],[500,510],  color="black", zorder=-1)
    ax.plot([39.5,40],[510,520],  color="black", zorder=-1)
    ax.plot([39.5,40],[520, 530],  color="black", zorder=-1)

    ax.plot([50,50.5],[355,363],  color="black", zorder=-1)
    ax.plot([50.3,50.8],[355,363],  color="black", zorder=-1)
    ax.plot([50.6,51.1],[355, 363],  color="black", zorder=-1)


    ax.set_ylabel("LEBENSZYKLUSKOSTEN in $10^3~$€")
    ax.legend()
    ax.set_xlabel("MAX. ZULÄSSIGER SCHALLDRUCKPEGEL in dB")
    ax.set_ylim(300, 600)
    ax.set_xlim(left=39,right=55)

    fig.tight_layout()
    if save_plots:
        outname = "pareto_noise_off"
        filedata = [y for x in standard_dict_list for y in x["filename"]]
        caption = r"Büroturm -- Konfigurationsoptimierung\\Pareto-Front der konfliktären Ziele akustischer Komfort und Lebenszykluskosten, dargestellt für die Regelstrategien \gls{cav}, \gls{vavvpc} und \gls{dfcpc}. Die Lebenszykluskosten steigen nur gering mit steigender akustischer Behaglichkeit. Keine Regelstrategie findet für maximal zulässige Schalldruckpegel von unter \SI{40}{\decibel} zulässige Lösungen."
        save_data(fig, serializer, outname, filedata, caption)

## Ist Integration der Akustik wirklich notwendig?
Wird überprüft, indem zweimal optimiert wird. Einmal wird die Konfiguration (also Entscheidung über Regelstrategie und gekauften Ventilatoren) aus der Topologieoptimierung übernommen. Bei der zweiten Optimierung ist die Konfiguration frei.
Wenn beide Optimierungen identisch sind, so hätte man die Akustikoptimierung nicht durchführen müssen.

In [ ]:
standard_case_bar_chart = True
if standard_case_bar_chart:

    folderpath_fix = "results/off/combined/max_velocity_5_10ms_max_height_04m/standard_case/"
    folderpath_var = "results/off/combined/max_velocity_5_10ms_max_height_04m/standard_case_configopt_no_fixed_configuration/"

    foldernames = ["CAV", "VAV-CPC", "VAV-VPC", "DF-CPC", "ONLY-DF", "ODS-CC"]

    folderpath_fix = [folderpath_fix + cs for cs in foldernames]
    folderpath_var = [folderpath_var + cs for cs in foldernames]

    standard_dict_list_fix = create_standard_list(folderpath_fix, "Configuration", use_connected_opt=False)
    standard_dict_list_var = create_standard_list(folderpath_var, "Configuration", use_connected_opt=False)


    # standard_dict_list = sort_list_of_dicts(standard_dict_list, "max_noise_value")
    standard_dict_list_fix = merge_standard_list(standard_dict_list_fix)
    standard_dict_list_var = merge_standard_list(standard_dict_list_var)


    colors = ['#000000ff', '#535353ff', '#898989ff', '#b5b5b5ff', '#dcdcdcff', "green"]

    # Plot setup
    serializer = MatplotlibSerializer()
    fig,ax = serializer.subplots(figsize=(diss_tex_width,diss_tex_width*5/9))

    entry_names = ["duct_costs", "equipment_investment_cost", "sil_costs", "vfc_costs", "fan_costs", "exact_fan_energy_costs"]
    cost_label = ["Kanal", "Regel-Eq.", "Schalldämpfer", "VSR", "Ventilator", "Energiekosten"]


    # Compute exact fan energy costs
    standard_dict_list_fix["exact_fan_energy_costs"] = (
        standard_dict_list_fix["fan_energy_costs"]
        * standard_dict_list_fix["exact_power_consumption"]
        / standard_dict_list_fix["power_consumption"]
    )
    # Compute exact fan energy costs
    standard_dict_list_var["exact_fan_energy_costs"] = (
        standard_dict_list_var["fan_energy_costs"]
        * standard_dict_list_var["exact_power_consumption"]
        / standard_dict_list_var["power_consumption"]
    )

    # for idx in range(len(curr_dict["exact_fan_energy_costs"])):
    #     costs = [curr_dict[name][idx] for name in entry_names]

        # Plot stacked bar
    for idx in range(len(standard_dict_list_fix["fan_energy_costs"])):
        costs_fix = [standard_dict_list_fix[name][idx]/1e3 for name in entry_names]
        costs_var = [standard_dict_list_var[name][idx]/1e3 for name in entry_names]
        # Plot stacked bar
        b_cost_fix, b_cost_var = 0, 0
        for i, (cost_fix, cost_var) in enumerate(zip(costs_fix, costs_var)):
            if i == len(costs_fix) - 1:
                ax.bar(idx-0.2, cost_fix, bottom=b_cost_fix, color="white", label=cost_label[i], hatch="xx", edgecolor="black", linewidth=0.3, width=.4)
                ax.bar(idx+0.2, cost_var, bottom=b_cost_var, color="white", hatch="xx", edgecolor="black", linewidth=0.3, width=.4)
            else:
                ax.bar(idx-0.2, cost_fix, bottom=b_cost_fix, color=colors[i], label=cost_label[i], edgecolor="black", linewidth=0.3, width=.4)
                ax.bar(idx+0.2, cost_var, bottom=b_cost_var, color=colors[i], edgecolor="black", linewidth=0.3, width=.4)
            b_cost_fix += cost_fix
            b_cost_var += cost_var

    # Optional: Customize axes
    ax.set_ylabel("LEBENSZYKLUSKOSTEN in $10^3~$€")
    # ax.set_xlabel("MAX. ZULÄSSIGER SCHALLDRUCKPEGEL in dB")

    # x_ticks = [str(f"{x:.1f}") for x in curr_dict["max_noise_value"]]
    
    ticknames = control_strategies_names_ger_twocolumn

    
    ax.set_xticks(range(len(standard_dict_list_fix["computation_time"])), ticknames, fontsize=9)

    ax = add_legend_right(ax, cost_label)
    
    fig.tight_layout()

## Variation der Kanalhöhe und Geschwindigkeit

In [ ]:
standard_case_bar_chart = True
if standard_case_bar_chart:

    cs = "CAV"

    folders = [
        "results/off/combined/duct_variations/max_velocity_5_10ms_max_height_0.5m/%s"%cs,
        "results/off/combined/duct_variations/max_velocity_5_10ms_max_height_0.4m/%s"%cs,
        "results/off/combined/duct_variations/max_velocity_5_10ms_max_height_0.3m/%s"%cs,
        "results/off/combined/duct_variations/max_velocity_5_10ms_max_height_0.2m/%s"%cs,
        ]

    names = ["0.5", "0.4", "0.3", "0.2"]

    standard_dict_list = create_standard_list(folders, "Topology", use_connected_opt=True)

    # standard_dict_list = sort_list_of_dicts(standard_dict_list, "max_noise_value")
    curr_dict = merge_standard_list(standard_dict_list)
    

    colors = ['#000000ff', '#535353ff', '#898989ff', '#b5b5b5ff', '#dcdcdcff', "green"]

    # Plot setup
    serializer = MatplotlibSerializer()
    fig,ax = serializer.subplots(figsize=(diss_tex_width,diss_tex_width*5/9))
    
    entry_names = ["duct_costs", "equipment_investment_cost", "sil_costs", "vfc_costs", "fan_costs", "exact_fan_energy_costs"]
    cost_label = ["Kanal", "Regel-Eq.", "Schalldämpfer", "VSR", "Ventilator     ", "Energiekosten"]

    # Compute exact fan energy costs
    curr_dict["exact_fan_energy_costs"] = (
        curr_dict["fan_energy_costs"]
        * curr_dict["exact_power_consumption"]
        / curr_dict["power_consumption"]
    )
    for idx in range(len(curr_dict["exact_fan_energy_costs"])):
        costs = [curr_dict[name][idx]/1e3 for name in entry_names]

        # Plot stacked bar
        b_cost = 0
        for i, cost in enumerate(costs):
            if i == len(costs) - 1:
                ax.bar(names[idx], cost, bottom=b_cost, color="white", label=cost_label[i], hatch="xx", edgecolor="black", linewidth=0.3)
            else:
                ax.bar(names[idx], cost, bottom=b_cost, color=colors[i], label=cost_label[i], edgecolor="black", linewidth=0.3)
            b_cost += cost

    # Optional: Customize axes
    ax.set_ylabel("LEBENSZYKLUSKOSTEN in $10^3~$€")

    ax.set_xlabel("MAXIMALE KANALHÖHE in m")
    ax = add_legend_right(ax, entry_names)

    # ax.set_ylim(bottom=560)

    fig.tight_layout()
    if save_plots:
        outname = "bar_chart_max_height_off"
        filedata = [str(x) for x in curr_dict["filename"]]
        caption = r"Büroturm -- Topologieoptimierung\\Vergleich der Lebenszykluskosten-optimalen Lösungen für verschiedene maximale Kanalhöhen. Die Lebenszykluskosten steigen für niedrigere Kanalhöhen nur geringfügig an. Für eine maximale Kanalhöhe von \SI{0.2}{\meter} hat das Optimierungsproblem keine Lösung mehr."
        save_data(fig, serializer, outname, filedata, caption)

In [ ]:
standard_case_bar_chart = True
if standard_case_bar_chart:

    cs = "VAV-VPC"

    folders = [
        "results/off/combined/duct_variations/max_velocity_5_5ms_max_height_0.4m/%s"%cs,
        "results/off/combined/duct_variations/max_velocity_8_8ms_max_height_0.4m/%s"%cs,
        "results/off/combined/duct_variations/max_velocity_9_9ms_max_height_0.4m/%s"%cs,
        "results/off/combined/duct_variations/max_velocity_10_10ms_max_height_0.4m/%s"%cs,
        "results/off/combined/duct_variations/max_velocity_12_12ms_max_height_0.4m/%s"%cs,
        ]

    names = ["5", "8", "9", "10", "12"]

    standard_dict_list = create_standard_list(folders, "Topology", use_connected_opt=True)

    # standard_dict_list = sort_list_of_dicts(standard_dict_list, "max_noise_value")
    curr_dict = merge_standard_list(standard_dict_list)
    

    colors = ['#000000ff', '#535353ff', '#898989ff', '#b5b5b5ff', '#dcdcdcff', "green"]

    # Plot setup
    serializer = MatplotlibSerializer()
    fig,ax = serializer.subplots(figsize=(diss_tex_width,diss_tex_width*5/9))
    
    entry_names = ["duct_costs", "equipment_investment_cost", "sil_costs", "vfc_costs", "fan_costs", "exact_fan_energy_costs"]
    cost_label = ["Kanal", "Regel-Eq.", "Schalldämpfer", "VSR", "Ventilator     ", "Energiekosten"]

    # Compute exact fan energy costs
    curr_dict["exact_fan_energy_costs"] = (
        curr_dict["fan_energy_costs"]
        * curr_dict["exact_power_consumption"]
        / curr_dict["power_consumption"]
    )
    for idx in range(len(curr_dict["exact_fan_energy_costs"])):
        costs = [curr_dict[name][idx]/1e3 for name in entry_names]

        # Plot stacked bar
        b_cost = 0
        for i, cost in enumerate(costs):
            if i == len(costs) - 1:
                ax.bar(names[idx], cost, bottom=b_cost, color="white", label=cost_label[i], hatch="xx", edgecolor="black", linewidth=0.3)
            else:
                ax.bar(names[idx], cost, bottom=b_cost, color=colors[i], label=cost_label[i], edgecolor="black", linewidth=0.3)
            b_cost += cost

    # Optional: Customize axes
    ax.set_ylabel("LEBENSZYKLUSKOSTEN in $10^3~$€")
   
    ax.set_xlabel("MAXIMALE KANALHÖHE in m")
    ax = add_legend_right(ax, entry_names)

    # ax.set_ylim(bottom=560)

    fig.tight_layout()
    if save_plots:
        outname = "bar_chart_max_height_off"
        filedata = [str(x) for x in curr_dict["filename"]]
        caption = r"Büroturm -- Topologieoptimierung\\Vergleich der Lebenszykluskosten-optimalen Lösungen für verschiedene maximale Kanalhöhen. Die Lebenszykluskosten steigen für niedrigere Kanalhöhen nur geringfügig an. Für eine maximale Kanalhöhe von \SI{0.2}{\meter} hat das Optimierungsproblem keine Lösung mehr."
        save_data(fig, serializer, outname, filedata, caption)

In [ ]:
standard_case_bar_chart = True
if standard_case_bar_chart:

    folders = [
        #"results/GPZ/combined/max_height_04m_no_velocity_limit/VAV-VPC/",
        "results/off/combined/duct_variations/max_velocity_5_5ms_max_height_0.4m/VAV-VPC/",
        "results/off/combined/duct_variations/max_velocity_8_8ms_max_height_0.4m/VAV-VPC/",
        "results/off/combined/duct_variations/max_velocity_10_10ms_max_height_0.4m/VAV-VPC/",
        "results/off/combined/duct_variations/max_velocity_12_12ms_max_height_0.4m/VAV-VPC/",
        ]

    # "FIXED FANS, VAR",

    names = ["5","8","10","12"]

    standard_dict_list = create_standard_list(folders, "Topology", use_connected_opt=True)

    # standard_dict_list = sort_list_of_dicts(standard_dict_list, "max_noise_value")
    curr_dict = merge_standard_list(standard_dict_list)


    colors = ['#000000ff', '#535353ff', '#898989ff', '#b5b5b5ff', '#dcdcdcff', "green"]

    # Plot setup
    serializer = MatplotlibSerializer()
    fig,ax = serializer.subplots(figsize=(diss_tex_width,diss_tex_width*5/9))
    


    for idx in range(len(curr_dict["mean_horizontal_velocity"])):
        t_bar = 0.2
        ax.bar(idx+t_bar, curr_dict["mean_vertical_velocity"][idx], color="k", width=t_bar*2)
        ax.bar(idx-t_bar, curr_dict["mean_horizontal_velocity"][idx], color="grey", width=t_bar*2)
        
    ax.set_xlabel("MITTLERE KANALGESCHWINDIGKEITEN in m/s")
    
    ax.legend(["Hauptkanal", "Abzweig"])

    ax.set_xticks(range(len(curr_dict["mean_horizontal_velocity"])), names)

    fig.tight_layout()

## Error analysis

### Energetically

In [ ]:
optimality_gaps = True
if optimality_gaps:
    
    foldernames = ["CAV", "VAV-CPC", "VAV-VPC", "DF-CPC", "ONLY-DF", "ODS-CC"]

    cs_names = foldernames

    folder = "results/off/combined/max_velocity_5_10ms_max_height_04m/standard_case/"

    ticknames = control_strategies_names_ger_twocolumn

    foldernames = ["CAV", "VAV-CPC", "VAV-VPC", "DF-CPC", "ONLY-DF", "ODS-CC"]
    filename_list = [folder + x for x in foldernames]

    standard_dict_list = create_standard_list(filename_list, "Topology", use_connected_opt=False)

    # Assume colors are in hex RGBA
    colors = ['#000000ff', '#535353ff', '#898989ff', '#b5b5b5ff',  '#dcdcdcff']

    # Compute indices of maximum LCC
    arg_max_lcc, _ = find_strategy_arg_min_max(standard_dict_list, "exact_lcc")

    # Plot setup
    serializer = MatplotlibSerializer()

    fig,ax = serializer.subplots(figsize=(diss_tex_width*0.8,diss_tex_width*0.8*5/9))

    entry_names = ["primal solution MINLP", "primal solution MIP", "dual bound MIP"]

    # Iterate over strategies
    for idx, curr_dict in enumerate(standard_dict_list):

        opt_idx = arg_max_lcc[idx]
        
        # Compute exact fan energy costs
        lcc_cost = curr_dict["fan_energy_costs"][opt_idx]/1e3 + curr_dict["invest_costs"][opt_idx]/1e3
        dual_bound_mip = (1-curr_dict["optimality_gap"][opt_idx]/100)*(lcc_cost)
        primal_bound_mip = lcc_cost
        primal_bound_minlp = curr_dict["exact_lcc"][opt_idx]/1e3

        mip_gap = lcc_cost - dual_bound_mip
        approx_error = primal_bound_minlp - lcc_cost

        ax.errorbar(control_strategies_names_ger_twocolumn[idx], primal_bound_minlp, yerr=[[mip_gap],[0]], fmt='o', markersize=3, capsize=5, color= "k")

        ax.errorbar(control_strategies_names_ger_twocolumn[idx], primal_bound_minlp, yerr=[[approx_error], [0]], fmt='o', markersize=3, capsize=5, color= "k")

    ax.set_ylabel("LEBENSZYKLUSKOSTEN\nin $10^3~$€")

    ax.errorbar(3.5, 300, yerr=[[50],[0]], fmt='o', markersize=3, capsize=5, color= "k")
    ax.errorbar(3.5, 300, yerr=[[100],[0]], fmt='o', markersize=3, capsize=5, color= "k")
    ax.text(3.7, 300, "primale Lösung MINLP", va="center", fontsize=8)
    ax.text(3.7, 250, "primale Lösung MILP", va="center", fontsize=8)
    ax.text(3.7, 200, "duale Lösung MILP", va="center", fontsize=8)
    
    ax.set_xticklabels(control_strategies_names_ger_twocolumn, fontsize=8)
    ax.set_ylim(bottom=0)

    fig.tight_layout()

# Computation times


In [ ]:
optimality_gaps = True
if optimality_gaps:

    foldernames = ["CAV", "VAV-CPC", "VAV-VPC", "DF-CPC", "ONLY-DF", "ODS-CC"]
    folder1 = "results/off/combined/max_velocity_5_10ms_max_height_04m/standard_case/"
    folder2 = "results/off/combined/max_velocity_5_10ms_max_height_04m/standard_case_configopt_no_fixed_configuration/"

    filename_list1 = [folder1 + x for x in foldernames]
    filename_list2 = [folder2 + x for x in foldernames]


    standard_dict_list1 = create_standard_list(filename_list1, "Topology", use_connected_opt=False)
    standard_dict_list2 = create_standard_list(filename_list2, "Configuration", use_connected_opt=False)

    # Plot setup
    serializer = MatplotlibSerializer()

    fig,ax = serializer.subplots(figsize=(diss_tex_width*0.8,diss_tex_width*0.8*5/9))

    # Iterate over strategies
    for idx, (curr_dict_topo, curr_dict_conf) in enumerate(zip(standard_dict_list1, standard_dict_list2)):
        if len(curr_dict_topo["computation_time"]) != 1 or len(curr_dict_conf["computation_time"]) != 1:
            raise ValueError("Too many results per control strategy")
        ax.bar(idx-0.15, curr_dict_topo["computation_time"][0], color= "grey", width=0.3)
        ax.bar(idx+0.15, curr_dict_conf["computation_time"][0], color= "black", width=0.3)

    ax.set_ylabel("RECHENZEIT in s")

    ax.set_xticks(np.arange(len(standard_dict_list1)), control_strategies_names_ger_twocolumn, fontsize=8)

    ax.legend(["Topologieoptimierung","Konfigurationsoptimierung"], fontsize=8, prop=arial_font)

    ax.set_yscale("log")

    fig.tight_layout()

    if save_plots:
        outname = "computation_time_off_topo_config"
        filedata = [*[x["filename"][0] if x["filename"] else "no file" for x in standard_dict_list1], *[x["filename"][0] if x["filename"] else "no file" for x in standard_dict_list2]]
        caption = r"Büroturm\\Rechenzeiten der Regelstrategien für Topologie- und holistische Konfigurationsoptimierung."
        save_data(fig, serializer, outname, filedata, caption)